In [1]:
START_DATE = "03/2020"
END_DATE = "09/2025"

In [2]:
import datetime
from pathlib import Path

from nwec.utility_reporting.analysis import powerpoint_utils

In [3]:
start_dt = datetime.datetime.strptime(START_DATE, "%m/%Y").replace(tzinfo=datetime.UTC)
end_dt = datetime.datetime.strptime(END_DATE, "%m/%Y").replace(tzinfo=datetime.UTC)

In [4]:
report_path = powerpoint_utils.copy_template(start_dt, end_dt)

In [5]:
import datetime

from pptx import Presentation

from nwec.constants import PROCESSED_UTILITY_DATA

start_month = start_dt.strftime("%B")
end_month = end_dt.strftime("%B")
start_year = start_dt.year
end_year = end_dt.year
date_range = ""

if start_year != end_year:
    date_range = f"{start_month} {start_year} - {end_month} {end_year}"
else:
    date_range = f"{start_month} - {end_month} {start_year}"
# Open the presentation
prs = Presentation(str(report_path))

# Get the slides
title_slide = prs.slides[0]
arrearage_counts_slide = prs.slides[4]
arrearage_amounts_slide = prs.slides[5]
arrearage_vintage_slide = prs.slides[6]
bill_assistance_slide = prs.slides[9]
disconnection_slide = prs.slides[15]

for shape in title_slide.shapes:
    if hasattr(shape, "text_frame"):
        for paragraph in shape.text_frame.paragraphs:
            for run in paragraph.runs:
                if "{DATE_RANGE}" in run.text:
                    run.text = run.text.replace("{DATE_RANGE}", date_range)

powerpoint_utils.add_centered_image(prs, arrearage_counts_slide, Path("graphs/arrearage_counts_stacked.png"))
powerpoint_utils.add_top_left_center_image(prs, arrearage_amounts_slide, Path("graphs/arrearage_amounts_stacked.png"))
powerpoint_utils.add_top_left_center_half_image(
    prs, arrearage_vintage_slide, Path("graphs/arrearage_amounts_vintage.png")
)
powerpoint_utils.add_top_right_center_half_image(
    prs, arrearage_vintage_slide, Path("graphs/kli_arrearage_amounts_vintage.png")
)
powerpoint_utils.add_centered_image(prs, bill_assistance_slide, Path("graphs/bill_assist_stacked.png"))
powerpoint_utils.add_centered_image(prs, disconnection_slide, Path("graphs/disconnections_stacked.png"))

In [6]:
prs.save(str(report_path))